# Imports

In [2]:
import sys
import os
from pathlib import Path

# Add parent directory to sys.path so hvac module is discoverable
# (notebook runs from blog_posts/, need to go up one level to anomaly_detection/)
sys.path.insert(0, os.path.dirname(os.getcwd()))

from hvac.utils import hvac_data_gen as hvdg
from datetime import datetime, timedelta
import plotly.express as px
import stumpy
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from hvac.utils import visual as vis
from hvac.utils import euclidean_dist as eucl_dist 

# Euclidean Distance model poor performance on Lag anomaly

## Load Data

In [3]:

hvac_dataset = pd.read_parquet(Path("../datasets/hvac_anomalies_v021226.parquet"))

fig1 = vis.plot_container_anomaly_timeseries(hvac_dataset, anomaly_type="lag", num_containers=1)
fig1.show()

fig2 = vis.plot_anomaly_type_distribution(hvac_dataset)
fig2.show()

In [37]:
hvac_dataset["day"] = hvac_dataset["timestamp_et"].dt.date                                                                                                                                                    
hvac_dataset = hvac_dataset[hvac_dataset['anomaly_type'].isin(['normal', 'lag'])].reset_index()


In [43]:
pivot_df = hvac_dataset.pivot_table(
    index=['container_id', 'timestamp_et'],
    columns='unit',
    values='anomaly_type',
    aggfunc='first'
)
# drop nans which arise due to filtering to non-lag anomalies in hvac_dataset
pivot_df = pivot_df.dropna()

In [ ]:
# For each timestamp, determine consensus anomaly type:
# If all units are normal → 'normal', else → first non-normal anomaly type
def get_consensus_anomaly(row):
    non_normal = row[row != 'normal']
    return 'normal' if len(non_normal) == 0 else non_normal.iloc[0]

ts_labels = pivot_df.apply(get_consensus_anomaly, axis=1).reset_index(name='type')

In [48]:
pivot_df.head()

unit                                   0       1       2
container_id timestamp_et                               
0            2026-01-01 00:00:00  normal  normal  normal
             2026-01-01 00:01:00  normal  normal  normal
             2026-01-01 00:02:00  normal  normal  normal
             2026-01-01 00:03:00  normal  normal  normal
             2026-01-01 00:04:00  normal  normal  normal

In [66]:

anomalous_df = pivot_df[pivot_df != 'normal'].dropna(how='all')
anomalous_df = anomalous_df.stack().dropna().reset_index(name='anomaly_type')
anomalous_df.head()


,container_id,timestamp_et,unit,anomaly_type
0,13,2026-01-04 12:00:00,2,lag
1,13,2026-01-04 12:01:00,2,lag
2,13,2026-01-04 12:02:00,2,lag
3,13,2026-01-04 12:03:00,2,lag
4,13,2026-01-04 12:04:00,2,lag


In [72]:

normal_df = pivot_df[pivot_df == 'normal'].dropna(how='any')
normal_df = normal_df.stack().reset_index(name='anomaly_type').drop_duplicates(subset=['container_id', 'timestamp_et'])
ts_labels_df = pd.concat([anomalous_df, normal_df]).sort_values(by=['container_id', 'timestamp_et'])
ts_labels_df.head()

,container_id,timestamp_et,unit,anomaly_type
0,0,2026-01-01 00:00:00,0,normal
3,0,2026-01-01 00:01:00,0,normal
6,0,2026-01-01 00:02:00,0,normal
9,0,2026-01-01 00:03:00,0,normal
12,0,2026-01-01 00:04:00,0,normal


In [74]:
ts_labels_df.shape

(9923160, 4)

In [77]:
ts_labels_df.duplicated(subset=['container_id', 'timestamp_et']).sum()

0

In [53]:
anomalous_df.notna().idxmax(axis=1)

container_id  timestamp_et       
13            2026-01-04 12:00:00    2
              2026-01-04 12:01:00    2
              2026-01-04 12:02:00    2
              2026-01-04 12:03:00    2
              2026-01-04 12:04:00    2
                                    ..
998           2026-01-06 14:55:00    2
              2026-01-06 14:56:00    2
              2026-01-06 14:57:00    2
              2026-01-06 14:58:00    2
              2026-01-06 14:59:00    2
Length: 127140, dtype: int64

In [ ]:

day_labels = (
      hvac_dataset.groupby(["container_id", "day"]).apply(
        lambda grp: pd.Series({
                'anomaly': grp['anomaly'].max(),
                'type': 'normal' if (grp['anomaly_type'] == 'normal').all() else (grp.loc[grp['anomaly_type'] != 'normal', 'anomaly_type']).unique()[0]
            })
      )
      
    #   .reset_index(name="label")
  )

# filter to lag anomaly
day_labels = day_labels[day_labels['type'].isin(['normal', 'lag'])]

## Create Model

In [5]:
dist_df = eucl_dist.compute_pairwise_distances(hvac_dataset)
eucl_scores_mean_df = eucl_dist.score_anomalies(dist_df, day_agg="mean")
eucl_scores_max_df = eucl_dist.score_anomalies(dist_df, day_agg="max")

## Visualize scores separation

In [6]:

eucl_scores_mean_labels_df = eucl_scores_mean_df.merge(day_labels.reset_index(), on=["container_id", "day"], how="left")
eucl_scores_max_labels_df = eucl_scores_max_df.merge(day_labels.reset_index(), on=["container_id", "day"], how="left")

In [7]:
fig = px.histogram(eucl_scores_mean_labels_df, x='anomaly_score', color='type', log_y=True, title="Day-level scores (mean agg)")
fig.show()
fig = px.histogram(eucl_scores_max_labels_df, x='anomaly_score', color='type', log_y=True, title="Day-level scores (max agg)")
fig.show()

## Timestamp-level scores: does finer granularity preserve anomaly signal?

Hypothesis: day-level aggregation dilutes short anomalies (e.g. 6h lag in 24h day). Compare ts-level score distributions to day-level above.

In [8]:
# Compute timestamp-level scores
eucl_scores_ts_df = eucl_dist.score_anomalies_ts(dist_df)


In [9]:
eucl_scores_ts_df.head()

,container_id,timestamp_et,anomaly_score,model
0,0,2026-01-01 01:08:00,-0.401781,euclidean_distance_ts_mad
1,0,2026-01-01 01:09:00,-0.400419,euclidean_distance_ts_mad
2,0,2026-01-01 01:10:00,-0.399092,euclidean_distance_ts_mad
3,0,2026-01-01 01:11:00,-0.397852,euclidean_distance_ts_mad
4,0,2026-01-01 01:12:00,-0.396600,euclidean_distance_ts_mad


In [ ]:
# Build timestamp-level labels by pivoting units as columns
# Pivot: rows are (container_id, timestamp_et), columns are unit IDs, values are anomaly_type


In [28]:
pivot_df.head()

unit                                   0       1       2
container_id timestamp_et                               
0            2026-01-01 00:00:00  normal  normal  normal
             2026-01-01 00:01:00  normal  normal  normal
             2026-01-01 00:02:00  normal  normal  normal
             2026-01-01 00:03:00  normal  normal  normal
             2026-01-01 00:04:00  normal  normal  normal

In [34]:
pivot_df[0].unique()

<ArrowStringArray>
['normal', nan, 'lag']
Length: 3, dtype: str

In [31]:
pivot_df[pivot_df != 'normal'].dropna()

,unit,0,1,2
container_id,timestamp_et,,,


In [15]:
# Vectorized: mask 'normal' as NaN, then get first non-NaN per row
masked = pivot_df.mask(pivot_df == 'normal')
first_non_normal = masked.bfill(axis=1).iloc[:, 0].fillna('normal')

ts_labels = first_non_normal.reset_index(name='type')

# Filter to normal and lag anomalies only
ts_labels = ts_labels[ts_labels["type"].isin(["normal", "lag"])]

KeyboardInterrupt: 

In [ ]:
ts_labels.head()

In [ ]:

# Filter to normal and lag anomalies only
ts_labels = ts_labels[ts_labels["type"].isin(["normal", "lag"])]
ts_labels.head()

In [10]:
# Build timestamp-level labels (normal vs lag only)
# Group by container_id and timestamp_et: label as "normal" only if ALL units are normal, otherwise use the anomalous type
ts_labels = (
    hvac_dataset[["container_id", "timestamp_et", "anomaly_type"]]
    .groupby(["container_id", "timestamp_et"])
    .apply(
        lambda grp: pd.Series({
            'type': 'normal' if (grp['anomaly_type'] == 'normal').all() else grp.loc[grp['anomaly_type'] != 'normal', 'anomaly_type'].unique()[0]
        })
    )
    .reset_index()
)

# Filter to normal and lag anomalies only
ts_labels = ts_labels[ts_labels["type"].isin(["normal", "lag"])]

# Merge scores with labels
eucl_scores_ts_labels_df = eucl_scores_ts_df.merge(
    ts_labels[["container_id", "timestamp_et", "type"]].drop_duplicates(),
    on=["container_id", "timestamp_et"],
    how="inner",
)

print(f"Timestamp-level scores: {len(eucl_scores_ts_labels_df):,} rows")
print(eucl_scores_ts_labels_df["type"].value_counts())

KeyboardInterrupt: 

In [11]:
hvac_dataset.head()

,timestamp_et,unit,TmpRet,anomaly,anomaly_type,container_id,cont_unit_day,day
0,2026-01-01 00:00:00,0,51.356354,False,normal,0,2026-01-01_0_0,2026-01-01
1,2026-01-01 00:00:00,1,50.211001,False,normal,0,2026-01-01_0_1,2026-01-01
2,2026-01-01 00:00:00,2,51.151240,False,normal,0,2026-01-01_0_2,2026-01-01
3,2026-01-01 00:01:00,0,51.331707,False,normal,0,2026-01-01_0_0,2026-01-01
4,2026-01-01 00:01:00,1,50.217684,False,normal,0,2026-01-01_0_1,2026-01-01


In [ ]:
fig = px.histogram(
    eucl_scores_ts_labels_df, x="anomaly_score", color="type",
    log_y=True, title="Timestamp-level scores (Euclidean distance, MAD)",
    nbins=100,
)
fig.show()

# 

## Andrews curves